# Stock research — read-only notebook

Interactive scratchpad for exercising `research_stock`, the informational
research path in this project (`app/research.py` / the `research_stock`
tool in `app/tools.py`). This notebook is deliberately research-only —
it never touches `buy_stock`, so nothing here can place, confirm, or
resume an order.

`research_stock` fans out to three narrow sub-agent calls (fundamentals,
technicals, news/sentiment) run concurrently via a `ThreadPoolExecutor`,
then a synthesis call condenses their notes into one three-paragraph,
no-advice summary. This notebook checks that at three levels:

1. **Section 1** calls `app.research.research_stock` directly — no tool
   wrapper, no agent, no graph. Just the sub-agent fan-out and synthesis.
2. **Section 2** calls the LangChain `@tool`-wrapped version from
   `app/tools.py`, the same object the agent binds and invokes.
3. **Section 3** drives the full agent (`app/agent.py`) through the
   configured LLM, including the routing rule that treats a phrased
   trading question ("should I buy X") as a request to research X rather
   than a request to buy it.

Run cells top to bottom. If you edit `app/*.py`, re-run the import cell
(autoreload picks up changes automatically).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Ensure the project root (this notebook's directory) is importable as `app.*`
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.config import settings

print(f"LLM provider: {settings.llm_provider}")
print(f"Using mock IBKR data: {settings.use_mock_ibkr}")

## 1. Direct call — no tool wrapper, no agent

Calls `app.research.research_stock` directly. This is the lowest-level
check: three researcher sub-agents (fundamentals, technicals,
news/sentiment) run concurrently, then a synthesis pass combines their
notes into one three-paragraph summary.

In [ ]:
import time

from app.research import research_stock

start = time.time()
summary = research_stock("AAPL")
elapsed = time.time() - start

print(f"Elapsed: {elapsed:.1f}s\n")
print(summary)

### 1a. Sub-agents run concurrently, not sequentially

`research_stock` submits all three researcher calls to a
`ThreadPoolExecutor` before waiting on any of them. If that's actually
true, total elapsed time should look like roughly one LLM call plus the
synthesis call — not the sum of four sequential calls. This cell just
times a second ticker as a sanity check; compare the printed elapsed time
against how long a single `get_llm().invoke(...)` call takes on its own.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

from app.config import get_llm

start = time.time()
get_llm().invoke([SystemMessage(content="Reply with one word."), HumanMessage(content="Ping.")])
single_call_elapsed = time.time() - start

start = time.time()
research_stock("MSFT")
research_elapsed = time.time() - start

print(f"Single LLM call:        {single_call_elapsed:.1f}s")
print(f"Full research_stock:    {research_elapsed:.1f}s  (4 calls total: 3 concurrent + 1 synthesis)")

## 2. Tool-wrapped call

Calls the `@tool`-decorated `research_stock` from `app/tools.py` — the
same object `ALL_TOOLS` exposes to the agent. Invoking it through
`.invoke(...)` (rather than calling the underlying function directly)
exercises the LangChain tool-calling interface itself.

In [ ]:
from app.tools import research_stock as research_stock_tool

print(research_stock_tool.invoke({"symbol": "nvda"}))

## 3. Full agent, real LLM

`app/agent.py`'s `ask()` is what the CLI and FastAPI endpoint call. Since
`research_stock` doesn't interrupt (unlike `buy_stock`), each call here is
a single normal turn — no confirm/resume dance needed. Requires
`GROQ_API_KEY` in `.env`.

In [ ]:
import re

from groq import RateLimitError

from app.agent import ask


def ask_retry(question: str, thread_id: str, max_retries: int = 8) -> str:
    """ask() wrapped with rate-limit backoff. The shared Groq free tier
    used for this demo caps at 8000 tokens/minute, which a notebook that
    fires several tool-schema-heavy calls back to back can hit easily —
    this just waits out whatever cooldown Groq reports and retries."""
    for attempt in range(max_retries):
        try:
            return ask(question, thread_id)
        except RateLimitError as e:
            wait = 5.0
            match = re.search(r"try again in ([\d.]+)s", str(e))
            if match:
                wait = float(match.group(1)) + 1.0
            print(f"Rate limited, waiting {wait:.1f}s before retry {attempt + 1}/{max_retries}...")
            time.sleep(wait)
    raise RuntimeError("Exceeded retries waiting for Groq rate limit to clear")

### 3a. Direct research request

A plain "research TICKER" ask should route straight to the `research_stock`
tool and come back with the three-paragraph summary.

In [ ]:
thread_id_direct = "notebook-research-direct"
print(ask_retry("research TSLA", thread_id_direct))

### 3b. A trading-phrased question routes to research, not to buy_stock

The system prompt (`app/agent.py`) tells the model to treat
"should I buy X" / "is it a good time to sell X" as a request to research
X, not as a buy instruction — and to remind the user it isn't giving
advice afterward. This confirms that boundary holds against the real
model, not just in the prompt text. If this ever produced a `buy_stock`
interrupt instead of a research summary, `ask_retry` below would return a
confirmation message instead of a paragraph summary.

In [ ]:
thread_id_vague = "notebook-research-vague"
print(ask_retry("should I buy NVDA?", thread_id_vague))

### 3c. Follow-up question against the same thread

Because `ask()` uses a `MemorySaver` checkpointer keyed by `thread_id`, a
follow-up in the same thread should resolve against the prior research
turn without re-stating the ticker.

In [ ]:
print(ask_retry("what did you say about its recent price trend?", thread_id_vague))

### 3d. Ambiguous or missing ticker

Asking for research without a ticker should make the model ask a
clarifying question in plain text rather than guessing a symbol.

In [ ]:
thread_id_ambiguous = "notebook-research-ambiguous"
print(ask_retry("can you research that stock for me?", thread_id_ambiguous))

## 4. Compare multiple tickers

Nothing special here — just a loop over `research_stock` (direct, not
through the agent) to eyeball a few summaries side by side. Useful for
spot-checking that the fundamentals/technicals/news split stays
consistent across tickers.

In [ ]:
for symbol in ["AAPL", "MSFT", "GOOGL"]:
    print(f"===== {symbol} =====")
    print(research_stock(symbol))
    print()